# HyDE (Hypothetical Document Embedding) 실험

HyDE(Hypothetical Document Embeddings)는 **사용자 질문에 대해 LLM(대형 언어 모델)로 가상의 문서를 먼저 생성하고, 이 문서를 임베딩하여 검색하는 방식**이다.  
기존에는 질문 자체를 임베딩해 유사한 문서를 찾았지만, HyDE는 질문의 맥락과 의도를 더 잘 반영할 수 있는 가상 문서를 생성해 임베딩하고, 이를 벡터 DB에서 비교함으로써 **더 정확한 검색 결과**를 얻을 수 있다.

**핵심 요약**
- 질문 → LLM이 가상 문서 생성 → 임베딩 → 벡터 DB에서 유사도 검색
- 기존 방식보다 **질문의 의미와 맥락을 더 잘 반영**하여 검색 정확도 향상
- RAG 등 다양한 검색·생성 AI 시스템에서 활용 가능
- 단점: LLM 사용으로 **속도가 느려질 수 있음**

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

# 벡터스토어 연결
vector_store = PineconeVectorStore(
    index_name= 'ir',   # 연결할 index명
    embedding= embeddings   # 사용할 임베딩 함수 (연결할 인덱스 차원과 임베딩 차원이 같아야 함
)

In [2]:
import pandas as pd

document_df = pd.read_csv('documents.csv')
queries_df = pd.read_csv('queries.csv')

queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=3;D4=1;D30=1
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3;D14=2;D26=1
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=3
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=3


In [3]:
# BM25 키워드 검색 모델
# - 문서를 토큰화해서 BM25 모델 생성
# - 검색어도 동일하게 토큰화하여 BM25 검색시 활용

from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi

kiwi = Kiwi()

def kiwi_tokenize(doc):
    return [token.form for token in kiwi.tokenize(doc)]

# 문서 내용 토큰화
tokenized_docs = [kiwi_tokenize(doc) for doc in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)    # BM25 모델 생성 (토큰화된 문서)

# 질의문을 토큰화해서 BM25로 상위 문서 검색하는 함수
def bm25_search(query,top_k = 5):
    query_tokens = kiwi_tokenize(query)         # 검색어 토큰화
    scores = bm25.get_scores(query_tokens)      # 각 문서의 BM25 점수 계산

    # 점수 기준 내림차순 정렬한 인덱스
    ranked_idx = sorted(range(len(scores)), key=lambda i :scores[i],reverse=True)
    # 상위 top_k개의 doc_id 리스트
    retrieved_docs = [document_df['doc_id'].iloc[i] for i in ranked_idx[:top_k]]
    return retrieved_docs

bm25_search('제주도 관광 명소')

['D1', 'D2', 'D3', 'D4', 'D5']

Hyde 가상문서 (가설답변) 생성 체인 구성

In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', temperature=0.3)
prompt = PromptTemplate.from_template('''
질문 : {query}

위 질문에 대해서 실제 문서가 아니어도 좋으니, 답변을 생성해주세요.
''')
output_parser = StrOutputParser()

hyde_chain = prompt|llm|output_parser
print(hyde_chain.invoke('제주도 관광 명소'))

제주도에는 자연경관, 문화유산, 해변, 오름 등 다양한 관광 명소가 있습니다.

### 제주도 대표 관광 명소

1. **한라산 국립공원**
   - 제주도의 상징인 한라산을 만날 수 있는 곳입니다.
   - 성판악·관음사 탐방로를 통해 정상 등반이 가능하며, 짧은 산책 코스도 있습니다.

2. **성산일출봉**
   - 유네스코 세계자연유산으로 지정된 대표 명소입니다.
   - 정상에 오르면 바다와 주변 마을을 한눈에 볼 수 있으며, 일출 명소로 유명합니다.

3. **우도**
   - 성산항에서 배를 타고 갈 수 있는 작은 섬입니다.
   - 서빈백사, 검멀레해변, 우도봉 등이 유명하며 자전거나 전기차로 둘러보기 좋습니다.

4. **협재해수욕장**
   - 맑고 푸른 바다와 하얀 모래사장이 아름다운 해변입니다.
   - 비양도를 바라보며 산책하거나 해수욕을 즐기기 좋습니다.

5. **함덕해수욕장**
   - 수심이 비교적 얕고 해변 주변에 카페와 산책로가 잘 조성되어 있습니다.
   - 제주 동부 여행에서 방문하기 좋은 해변입니다.

6. **만장굴**
   - 세계적으로 유명한 용암동굴 중 하나입니다.
   - 독특한 지질 구조와 용암석주를 관찰할 수 있습니다.

7. **주상절리대**
   - 서귀포 중문 지역에 위치한 해안 절벽입니다.
   - 파도에 의해 만들어진 육각형 모양의 주상절리가 장관을 이룹니다.

8. **천지연폭포**
   - 서귀포의 대표적인 폭포로, 주변 산책로가 잘 조성되어 있습니다.
   - 밤에는 조명이 켜져 야경 명소로도 알려져 있습니다.

9. **오설록 티 뮤지엄**
   - 제주 녹차 문화를 체험할 수 있는 곳입니다.
   - 넓은 녹차밭과 카페, 전시 공간이 있어 사진 촬영 장소로도 인기가 많습니다.

10. **카멜리아힐**
    - 계절별로 다양한 꽃과 나무를 감상할 수 있는 수목원입니다.
    - 특히 동백꽃이 피는 계절에 아름답습니다.

11. **동문시장**
    - 제주 특산물과 해산물, 흑돼지 간식

In [17]:
import time 
from tqdm.auto import tqdm

hyde_pseudo = {}

for idx, row in tqdm(queries_df.iterrows()):
    qid = row['query_id']
    query_text = row['query_text']
    pseudo_answer = hyde_chain.invoke(query_text)   # 가상 답변 생성
    hyde_pseudo[qid] = pseudo_answer    # {qid : pseudo_answer,...}
    time.sleep(0.5)     # API 과호출 방지용 딜레이

hyde_pseudo

0it [00:00, ?it/s]

{'Q1': '제주 올레길은 코스마다 풍경과 난이도가 달라서 여행 일정과 체력에 맞춰 선택하는 것이 좋습니다. 처음 걷는다면 아래 코스를 추천합니다.\n\n## 1. 가장 무난한 해안 풍경: 올레길 6코스\n- **구간:** 쇠소깍~제주올레여행자센터\n- **거리:** 약 11km\n- **소요 시간:** 4~5시간\n- **난이도:** 쉬움\n- **주요 풍경:** 쇠소깍, 검은모래 해변, 정방폭포 주변, 서귀포 해안\n- **추천 이유:** 비교적 짧고 대중교통과 식당 이용이 편리합니다. 바다와 마을 풍경을 균형 있게 볼 수 있어 첫 올레길로 좋습니다.\n\n## 2. 제주다운 해안 절경: 올레길 7코스\n- **구간:** 외돌개~월평\n- **거리:** 약 17.7km\n- **소요 시간:** 5~6시간\n- **난이도:** 보통\n- **주요 풍경:** 외돌개, 황우지해안, 법환포구, 월평해안\n- **추천 이유:** 제주 올레길 중 가장 유명한 코스 중 하나입니다. 해안 절벽과 바다 풍경이 아름답고 사진 찍기 좋습니다.\n- **주의:** 전체 코스가 길게 느껴질 수 있어 체력이 부담되면 외돌개~법환포구 구간만 걸어도 좋습니다.\n\n## 3. 성산일출봉을 만나는 코스: 올레길 1코스\n- **구간:** 시흥초등학교~광치기해변\n- **거리:** 약 15km\n- **소요 시간:** 4~5시간\n- **난이도:** 보통\n- **주요 풍경:** 말미오름, 알오름, 성산일출봉, 섭지코지 인근 풍경\n- **추천 이유:** 오름과 바다를 한 번에 경험할 수 있습니다. 날씨가 맑으면 성산일출봉 주변 경관이 특히 아름답습니다.\n- **주의:** 오름 구간은 바람이 강하거나 미끄러울 수 있습니다.\n\n## 4. 바다와 송악산 풍경: 올레길 10코스\n- **구간:** 화순금모래해변~하모체육공원\n- **거리:** 약 15.6km\n- **소요 시간:** 5시간 안팎\n- **난이도:** 보통\n- **주요 풍경:** 산방산, 용머리해안 주변, 송악산, 모슬

In [18]:
import pandas as pd

pd.set_option('display.max_colwidth',None)  # 긴 텍스트도 생갹 없이 표시

hyde_df = pd.DataFrame({
    'query_id' : list(hyde_pseudo.keys()),  # qid 리스트
    'query_text': queries_df['query_text'], # 원본 질문 Series
    'pseudo_answer' : list(hyde_pseudo.values()) # 가상 질문 리스트
})

hyde_df.head()

,query_id,query_text,pseudo_answer
0,Q1,제주도 올레길 트레킹 코스 추천,"제주 올레길은 코스마다 풍경과 난이도가 달라서 여행 일정과 체력에 맞춰 선택하는 것이 좋습니다. 처음 걷는다면 아래 코스를 추천합니다.\n\n## 1. 가장 무난한 해안 풍경: 올레길 6코스\n- **구간:** 쇠소깍~제주올레여행자센터\n- **거리:** 약 11km\n- **소요 시간:** 4~5시간\n- **난이도:** 쉬움\n- **주요 풍경:** 쇠소깍, 검은모래 해변, 정방폭포 주변, 서귀포 해안\n- **추천 이유:** 비교적 짧고 대중교통과 식당 이용이 편리합니다. 바다와 마을 풍경을 균형 있게 볼 수 있어 첫 올레길로 좋습니다.\n\n## 2. 제주다운 해안 절경: 올레길 7코스\n- **구간:** 외돌개~월평\n- **거리:** 약 17.7km\n- **소요 시간:** 5~6시간\n- **난이도:** 보통\n- **주요 풍경:** 외돌개, 황우지해안, 법환포구, 월평해안\n- **추천 이유:** 제주 올레길 중 가장 유명한 코스 중 하나입니다. 해안 절벽과 바다 풍경이 아름답고 사진 찍기 좋습니다.\n- **주의:** 전체 코스가 길게 느껴질 수 있어 체력이 부담되면 외돌개~법환포구 구간만 걸어도 좋습니다.\n\n## 3. 성산일출봉을 만나는 코스: 올레길 1코스\n- **구간:** 시흥초등학교~광치기해변\n- **거리:** 약 15km\n- **소요 시간:** 4~5시간\n- **난이도:** 보통\n- **주요 풍경:** 말미오름, 알오름, 성산일출봉, 섭지코지 인근 풍경\n- **추천 이유:** 오름과 바다를 한 번에 경험할 수 있습니다. 날씨가 맑으면 성산일출봉 주변 경관이 특히 아름답습니다.\n- **주의:** 오름 구간은 바람이 강하거나 미끄러울 수 있습니다.\n\n## 4. 바다와 송악산 풍경: 올레길 10코스\n- **구간:** 화순금모래해변~하모체육공원\n- **거리:** 약 15.6km\n- **소요 시간:** 5시간 안팎\n- **난이도:** 보통\n- **주요 풍경:** 산방산, 용머리해안 주변, 송악산, 모슬포 해안\n- **추천 이유:** 제주 서남부의 대표적인 풍경을 볼 수 있는 코스입니다. 산방산과 송악산을 배경으로 걷는 길이 인상적입니다.\n- **주의:** 용머리해안은 기상과 물때에 따라 입장이 제한될 수 있습니다.\n\n## 5. 짧고 여유로운 숲길: 올레길 14-1코스\n- **구간:** 저지예술정보화마을~무릉외갓집\n- **거리:** 약 9km\n- **소요 시간:** 3~4시간\n- **난이도:** 쉬움~보통\n- **주요 풍경:** 곶자왈, 숲길, 밭길, 제주 중산간 마을\n- **추천 이유:** 바닷길보다 조용한 숲길을 선호한다면 좋습니다. 관광객이 비교적 적고 제주 중산간의 분위기를 느낄 수 있습니다.\n- **주의:** 숲길에서는 휴대전화 신호가 약한 구간이 있을 수 있으므로 이정표를 잘 확인해야 합니다.\n\n## 6. 제주 원도심과 해안 산책: 올레길 18코스\n- **구간:** 제주원도심~조천 만세동산\n- **거리:** 약 19km\n- **소요 시간:** 6시간 이상\n- **난이도:** 보통\n- **주요 풍경:** 제주항, 사라봉, 별도봉, 해안 마을, 조천 해안\n- **추천 이유:** 자연뿐 아니라 제주 도심과 역사적인 장소도 함께 볼 수 있습니다.\n- **주의:** 거리가 긴 편이라 하루 일정을 넉넉히 잡는 것이 좋습니다.\n\n## 여행 일정별 추천\n\n### 반나절만 걸을 때\n- 올레길 6코스 일부\n- 올레길 7코스 외돌개~법환포구\n- 올레길 9코스 대평포구 주변\n\n### 하루 동안 대표 풍경을 보고 싶을 때\n- 올레길 7코스\n- 올레길 10코스\n- 올레길 1코스\n\n### 조용한 길을 선호할 때\n- 올레길 14-1코스\n- 올레길 13코스\n- 올레길 18코스 일부\n\n### 대중교통을 이용할 때\n서귀포 중심 숙소라면 **6코스와 7코스**가 편리하고, 성산·모슬포 쪽 숙소라면 각각 **1코스와 10코스**를 선택하기 좋습니다. 코스 시작점과 종점의 버스 운행 간격이 길 수 있으므로 출발 전 교통편을 확인하는 것이 좋습니다.\n\n## 준비물과 주의사항\n- 미끄럽지 않은 트레킹화 또는 운동화\n- 물과 간단한 간식\n- 모자, 자외선 차단제, 우비\n- 바람막이 재킷\n- 휴대전화 보조배터리\n- 공식 올레길 앱 또는 코스 안내 지도\n\n제주도는 날씨가 빠르게 바뀌고 바람이 강한 날이 많습니다. 특히 오름과 해안 절벽 구간은 비가 온 뒤 미끄러울 수 있으므로 출발 전 날씨와 코스 통제 여부를 확인하세요.\n\n**처음 걷는다면 개인적으로는 올레길 6코스**, 제주다운 절경을 가장 많이 보고 싶다면 **7코스**, 성산일출봉까지 함께 보고 싶다면 **1코스**를 추천합니다."
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,"## 전주비빔밥 vs 진주비빔밥\n\n두 음식 모두 여러 가지 나물과 고기, 양념을 밥에 섞어 먹는 전통 비빔밥이지만, **전주비빔밥은 재료의 다양성과 고소하고 진한 양념 맛**, **진주비빔밥은 색감과 나물의 담백함, 육회의 풍미**가 상대적으로 두드러집니다. 다만 가게와 조리법에 따라 차이가 있을 수 있습니다.\n\n| 구분 | 전주비빔밥 | 진주비빔밥 |\n|---|---|---|\n| 대표 특징 | 다양한 나물과 육회, 황포묵을 올림 | 여러 색의 나물을 꽃처럼 담는 ‘칠보화반’ |\n| 주요 재료 | 콩나물, 시금치, 고사리, 도라지, 표고버섯, 애호박, 무나물, 육회, 황포묵, 달걀 등 | 숙주나물, 고사리, 도라지, 시금치, 무나물, 애호박, 육회 등 |\n| 밥 | 쌀밥 또는 쇠고기 육수·콩나물 등을 활용한 밥 | 비교적 담백한 밥에 여러 나물을 곁들이는 방식 |\n| 양념 | 고추장, 참기름, 깨, 마늘 등 | 고추장과 참기름을 사용하지만 재료 본연의 맛을 살리는 편 |\n| 곁들임 | 콩나물국이 자주 함께 나옴 | 선짓국이나 국물을 곁들이는 경우가 있음 |\n| 맛의 인상 | 고소하고 진하며 감칠맛이 풍부함 | 담백하고 깔끔하며 나물과 육회의 맛이 선명함 |\n\n### 1. 전주비빔밥의 재료와 맛\n\n전주비빔밥은 보통 다음과 같은 재료를 사용합니다.\n\n- 여러 가지 나물\n- 콩나물\n- 육회 또는 익힌 쇠고기\n- 황포묵\n- 달걀\n- 표고버섯\n- 고추장\n- 참기름과 깨\n\n전주비빔밥의 특징은 **재료의 종류가 많고 각각의 나물을 따로 정성스럽게 무친다**는 점입니다. 특히 콩나물과 황포묵이 들어가 아삭하거나 부드러운 식감이 함께 느껴집니다.\n\n맛은 대체로 **고소하고 진하며 감칠맛이 풍부한 편**입니다. 고추장과 참기름을 넣고 비비면 양념 맛이 전체적으로 잘 어우러지고, 육회나 쇠고기가 들어갈 경우 고기 특유의 풍미도 강하게 느껴집니다.\n\n### 2. 진주비빔밥의 재료와 맛\n\n진주비빔밥은 ‘**칠보화반**’이라고도 불립니다. 여러 가지 나물을 일곱 가지 색처럼 아름답게 배치해 담아내는 것이 특징입니다.\n\n주로 들어가는 재료는 다음과 같습니다.\n\n- 숙주나물\n- 고사리\n- 도라지\n- 시금치\n- 무나물\n- 애호박\n- 육회 또는 쇠고기\n- 달걀이나 김가루\n\n진주비빔밥은 나물의 색과 모양을 살려 담는 데 중점을 두기 때문에, 전주비빔밥보다 **재료 하나하나의 맛과 식감이 비교적 또렷하게 느껴지는 편**입니다.\n\n맛은 **담백하고 깔끔하며, 숙주나물과 각종 나물의 산뜻함이 돋보이는 편**입니다. 육회가 올라가면 부드럽고 고소한 맛이 더해지지만, 전주비빔밥처럼 전체적으로 양념 맛이 강하고 진하게 느껴지기보다는 나물과 고기의 조화를 강조하는 경우가 많습니다.\n\n## 간단히 정리하면\n\n- **전주비빔밥**: 재료가 다양하고 양념과 참기름의 맛이 진함. 고소하고 풍성한 맛.\n- **진주비빔밥**: 나물의 색과 식감을 살리고 육회의 풍미를 강조함. 담백하고 깔끔한 맛.\n- **전주비빔밥을 좋아할 사람**: 진한 고추장 양념과 다양한 재료를 한꺼번에 비벼 먹는 것을 좋아하는 사람.\n- **진주비빔밥을 좋아할 사람**

### Hyde(가상답변) 기반 Dense검색 결과 수집

In [23]:
hyde_results = {}

for idx, row in tqdm(queries_df.iterrows()):
    qid = row['query_id']
    query_text = row['query_text']
    pseudo_answer = row['pseudo_answer']
    docs = vector_store.similarity_search(pseudo_answer,k=5)
    hyde_results[qid] = [doc.metadat['doc_id']for doc in docs]

hyde_results

0it [00:00, ?it/s]

KeyError: 'pseudo_answer'

In [ ]:
# Vector Store 검색 결과를 쿼리별로 수집

dense_results = {}   # qid별 BM25 검색 결과 저장 dict
for idx, row in queries_df.iterrows():
    qid = row['query_id']   # 쿼리 ID 추출
    query_text = row['query_text']  # 쿼리 문장
     # {qid : query_text와 가장 가까운 5개의 doc_id}
    docs = vector_store.similarity_search(query_text, top_k = 5)  
    dense_results[qid] = [doc.metadata['doc_id']for doc in docs]
dense_results

In [24]:
import numpy as np

def parse_relevant(relevant_str) -> dict[str,int]:
    """ 
    참조문서 답안 문자열을 dict로 파싱하는 함수

    relevant_str = "D1=3;D4=1;D30=1" -> {'D1' :3, 'D2':1, 'D30':1}
    """

    pairs = relevant_str.split(';')
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split('=')
        rel_dict[doc_id] = int(grade)

    return rel_dict

def compute_metrics(predicted, relevant_dict, k=5) -> tuple[float,float,float,float]:
    hits = sum([1 for doc in predicted[:k] if doc in relevant_dict])
    precision = hits /k     # 정밀도 = 맞춘갯수 / 전체갯수

    # Recall
    total_relevant = len(relevant_dict) # 전체 관련 문서 수
    recall = hits / total_relevant if total_relevant > 0 else 0 # 재현율 = 관련 문서 맞춘 수

    # MRR 예측치 중 첫 관련문서 순위 점수
    rr = 0  # 관련 문서 수
    for idx,doc in enumerate(predicted):
        if doc in relevant_dict:
            rr=1/(idx+1)
            break   # 첫번째 적중 rr반영 후 반복문 탈출

    # AP (MAP를 위한 사전 계산)
    num_correct = 0 # 현재까지의 적중 횟수
    precisions = [] # 적중시의 precision
    for idx, doc in enumerate(predicted[:k]):   # top_k 범위에서
        if doc in relevant_dict:
            num_correct+=1  # 적중시 1 누적
            precisions.append(num_correct/(idx+1))  # 현재 시점의 precision 기록
    ap = np.mean(precisions) if precisions else 0

    return precision, recall, rr, ap

# 여러 쿼리에 대한 평균 성능지표 계산하는 함수
def evaluate_all(method_results,queries_df,k=5):
    prec_list,rec_list,rr_list,ap_list = [],[],[],[]    # 지표별 결과 저장

    for idx, row in queries_df.iterrows():
        qid = row['query_id']   # 쿼리 id
        relevant_dict = parse_relevant(row['relevant_doc_ids'])  # 정답 dict 파싱
        predicted = method_results[qid]     # 해당 쿼리의 예측 랭킹
        p,r,rr,ap = compute_metrics(predicted,relevant_dict,k)  # 지표 계산
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)

    return{
        'P@k':np.mean(prec_list),
        'R@k':np.mean(rec_list),
        'MRR':np.mean(rr_list),
        'MAP':np.mean(ap_list)
    }

In [25]:
# BM25/Dense 결과로 P@5, R@5, MRR, MAP 계산

bm25_metrics = evaluate_all(bm25_results, queries_df, k=5)
dense_metrics = evaluate_all(dense_results, queries_df, k=5)
hyde_metrics = evaluate_all(hyde_results, queries_df, k=5)


NameError: name 'bm25_results' is not defined

In [26]:
metrics_df = pd.DataFrame({
    'Metrics':['P@5', 'R@5', 'MRR', 'MAP'],
    'BM25' : [bm25_metrics['P@k'],bm25_metrics['R@k'],bm25_metrics['MRR'],bm25_metrics['MAP']],
    'Dense' : [dense_metrics['P@k'],dense_metrics['R@k'],dense_metrics['MRR'],dense_metrics['MAP']],
    'Hyde' : [hyde_metrics['P@k'],hyde_metrics['R@k'],hyde_metrics['MRR'],hyde_metrics['MAP']],
})
metrics_df

NameError: name 'bm25_metrics' is not defined

In [27]:
import matplotlib.pyplot as plt

metrics = ['P@5', 'R@5', 'MRR', 'MAP']
bm25_vals = [bm25_metrics['P@k'], bm25_metrics['R@k'], bm25_metrics['MRR'], bm25_metrics['MAP']]
dense_vals = [dense_metrics['P@k'], dense_metrics['R@k'], dense_metrics['MRR'], dense_metrics['MAP']]
hyde_vals = [hyde_metrics['P@k'], hyde_metrics['R@k'], hyde_metrics['MRR'], hyde_metrics['MAP']]

x = range(len(metrics))  # 0~3

plt.figure(figsize=(8, 4))
plt.plot(x, bm25_vals, marker='o', label='BM25')
plt.plot(x, dense_vals, marker='s', label='Dense')
plt.plot(x, hyde_metrics, marker='s', label='Hyde')
plt.xticks(x, metrics)  # x축 눈금은 0~3 -> 지표명
plt.ylim(0, 1.1)
plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('BM25 vs Dense vs Hyde Retrival')
plt.legend()
plt.grid()
plt.show()

NameError: name 'bm25_metrics' is not defined

##### HyDE의 역할과 효과
- HyDE는 쿼리를 “의미가 풍부한 가상문서”로 확장
- Dense가 이미 강력한 경우:
    - 성능을 보완하거나 유지
    - 항상 큰 폭의 개선을 보장하지는 않음
- 하지만 쿼리가 짧거나 모호한 경우, HyDE의 효과는 더욱 커질 가능성 있음

- Dense Retrieval은 의미 기반 검색의 기본 선택지
- HyDE는 다음 상황에서 특히 유용:
    - 짧고 추상적인 쿼리
    - 질문형 질의
    - Recall 개선이 중요한 경우
- 실서비스에서는: Dense + (HyDE / BM25 / Rerank) 조합이 가장 현실적인 접근